In [ ]:
!pip install SpeechRecognition deep-translator gTTS

In [ ]:
import speech_recognition as sr
from deep_translator import GoogleTranslator
from gtts import gTTS
import os
import time
import base64
from IPython.display import display, Javascript, Audio
from google.colab import output


# 🌐 UNLIMITED JAVASCRIPT AUDIO RECORDER WITH STOP BUTTON

RECORD_JS = """
var record = () => new Promise(async resolve => {
  let stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  let recorder = new MediaRecorder(stream);
  let chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);

  const div = document.createElement('div');
  const btn = document.createElement('button');
  btn.textContent = '⏹️ Stop Recording (Click when finished)';
  btn.style.padding = '12px 24px';
  btn.style.fontSize = '16px';
  btn.style.background = '#d9534f';
  btn.style.color = 'white';
  btn.style.border = 'none';
  btn.style.borderRadius = '5px';
  btn.style.cursor = 'pointer';
  btn.style.margin = '15px 0';
  btn.style.fontWeight = 'bold';

  div.appendChild(btn);
  document.body.appendChild(div);

  recorder.start();

  btn.onclick = () => {
    recorder.stop();
  };

  recorder.onstop = async () => {
    const blob = new Blob(chunks);
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
    div.remove();
  };
});
"""

def record_unlimited_audio():
    """Captures system audio streaming until user clicks the stop action"""
    display(Javascript(RECORD_JS))
    print("\n🎙️  === Microphone Active: Please start speaking... ===")

    s = output.eval_js('record()')
    b = base64.b64decode(s.split(',')[1])

    webm_filename = 'user_voice.webm'
    wav_filename = 'user_voice.wav'

    with open(webm_filename, 'wb') as f:
        f.write(b)

    os.system(f'ffmpeg -i {webm_filename} -ar 16000 -ac 1 -y {wav_filename} -loglevel quiet')

    if os.path.exists(webm_filename):
        os.remove(webm_filename)

    return wav_filename

def play_colab_audio(text, lang):
    """Generates text-to-speech output with auto-playback inside Colab"""
    tts = gTTS(text=text, lang=lang)
    filename = "translated_audio.mp3"
    tts.save(filename)
    display(Audio(filename, autoplay=True))


# 🚀 ONE-SHOT VOICE TRANSLATOR (PROFESSIONAL INTERFACE)

def main():
    print("Choose any option which you want...")
    print("1. Bangla to English (Speak Bangla -> Listen English)")
    print("2. English to Bangla (Speak English -> Listen Bangla)")

    choice = input("\nEnter your choice (1 or 2): ")

    if choice == '1':
        src_lang, tgt_lang, tts_lang = 'bn', 'en', 'en'
        recognizer_lang = 'bn-BD'
        print("\n[ Selected Mode: Bangla to English ]")
    elif choice == '2':
        src_lang, tgt_lang, tts_lang = 'en', 'bn', 'bn'
        recognizer_lang = 'en-US'
        print("\n[ Selected Mode: English to Bangla ]")
    else:
        print("❌ Invalid input! Please re-run the cell and select 1 or 2.")
        return

    try:
        r = sr.Recognizer()
        audio_file = record_unlimited_audio()

        print("\n⏳ Processing your voice, please wait...")

        with sr.AudioFile(audio_file) as source:
            audio_data = r.record(source)

        # 1. Speech-to-Text Engine Execution
        original_text = r.recognize_google(audio_data, language=recognizer_lang)

        # 2. Neural Translation Model Execution
        translated_text = GoogleTranslator(source=src_lang, target=tgt_lang).translate(original_text)

        # 3. Synchronized UI Output Display
        print("\n Output Display ")
        print(f"🗣️  [You Said]    : {original_text}")
        print(f"📝 [Translation] : {translated_text}")
        print("\n")
        print("🔊 Audio Output Playing...")


        play_colab_audio(translated_text, tts_lang)

        if os.path.exists(audio_file):
            os.remove(audio_file)

    except sr.UnknownValueError:
        print("\n❌ System could not recognize the voice. Please re-run the cell and try again.")
    except Exception as e:
        print(f"\n⚠️ Error: {e}")

if __name__ == "__main__":
    main()

Choose any option which you want...
1. Bangla to English (Speak Bangla -> Listen English)
2. English to Bangla (Speak English -> Listen Bangla)

Enter your choice (1 or 2): 2

[ Selected Mode: English to Bangla ]


<IPython.core.display.Javascript object>


🎙️  === Microphone Active: Please start speaking... ===

⏳ Processing your voice, please wait...

 Output Display 
🗣️  [You Said]    : he goes to outside
📝 [Translation] : সে বাইরে যায়


🔊 Audio Output Playing...
